In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-6 ; m=1; l0=1 ;l=2; k=2; w=2.0 ; g=9.8 ;  
tf=15; N=int(tf/dt);

t=np.linspace(0,tf,N)


theta_0= np.pi/3             #condiciones iniciales
w_0= 0                       # θ̇(0)

## LAGRANGIANO Y ECUACIONES DE MOVIMIENTO

In [ ]:
th_s, thd_s = sp.symbols('theta thetadot', real=True)
m_, l_, l0_, k_, g_, om_ = sp.symbols('m l l0 k g omega', positive=True)

T_sym = m_*l_**2*(thd_s**2 + om_**2*sp.sin(th_s)**2)                                    # 2 masas
U_sym = -2*m_*g_*l_*sp.cos(th_s) + sp.Rational(1,2)*k_*(2*l_*sp.cos(th_s) - l0_)**2     # gravedad + muelle
L_sym = T_sym - U_sym
E_sym = T_sym + U_sym

pth_sym = sp.diff(L_sym, thd_s)                         # p_θ = ∂L/∂θ̇   (derivada parcial)
H_sym   = sp.simplify(pth_sym*thd_s - L_sym)            # H = T2 - T0 + U   (integral de Jacobi)

# Euler-Lagrange:  d/dt(∂L/∂θ̇) = ∂L/∂θ   ->   θ̈
thdd_sym = sp.simplify( (sp.diff(L_sym, th_s) - sp.diff(pth_sym, th_s)*thd_s)/sp.diff(pth_sym, thd_s) )
print('θ̈ =', thdd_sym)
print('H  =', H_sym)

args = (th_s, thd_s, m_, l_, l0_, k_, g_, om_)
ath_num, L_num, E_num, H_num = (sp.lambdify(args, f, 'numpy') for f in (thdd_sym, L_sym, E_sym, H_sym))

## FUNCION PARA EL MOVIMIENTO

In [ ]:
def Motion(th0, thd0, pars, t):
    N = len(t)
    th, thd = np.zeros(N), np.zeros(N)
    th[0], thd[0] = th0, thd0

    for i in range(N - 1):
        dt = t[i+1] - t[i]

        # --- PREDICTOR (Euler) ---
        a_pred   = ath_num(th[i], thd[i], *pars)
        thd_pred = thd[i] + a_pred * dt
        th_pred  = th[i]  + thd[i] * dt

        # --- Aceleración en el punto predicho ---
        a_next = ath_num(th_pred, thd_pred, *pars)

        # --- CORRECTOR (Heun) ---
        th[i+1]  = th[i]  + 0.5 * (thd[i] + thd_pred) * dt
        thd[i+1] = thd[i] + 0.5 * (a_pred + a_next) * dt

    return th, thd


#Simulación
pars = (m, l, l0, k, g, w)
theta, thd = Motion(theta_0, w_0, pars, t)

## EQUILIBRIO Y PEQUEÑAS OSCILACIONES

In [ ]:
#Equilibrio:  dV/dθ = 0  con  V = U - T0 = U - m l² ω² sin²θ   (potencial efectivo en el sistema giratorio)
Veff_sym = U_sym - m_*l_**2*om_**2*sp.sin(th_s)**2
dV_sym   = sp.diff(Veff_sym, th_s)                       # = sinθ · G(cosθ)
c = sp.symbols('c', real=True)
G_sym = sp.simplify(dV_sym/sp.sin(th_s))
cos_eq_sym = sp.solve(G_sym.subs(sp.cos(th_s), c), c)[0]  # cosθ_eq
print('cos(θ_eq) =', cos_eq_sym)

d2V_num = sp.lambdify((th_s, m_, l_, l0_, k_, g_, om_), sp.diff(Veff_sym, th_s, 2), 'numpy')

c_eq = float(cos_eq_sym.subs({m_: m, l_: l, l0_: l0, k_: k, g_: g, om_: w}))
w_c2 = (m*g + k*l0)/(m*l) - 2*k/m                          # ω crítica²: existe θ_eq≠0 si ω² > w_c2
print(f'cos(θ_eq) = {c_eq:.4f}   |   ω = {w:.3f} rad/s,  ω_crítica = {np.sqrt(max(w_c2, 0)):.3f} rad/s')

if abs(c_eq) < 1:
    th_eq = np.arccos(c_eq)
    print(f'θ_eq = {th_eq:.5f} rad = {np.degrees(th_eq):.3f}°')
else:
    th_eq = 0.0
    print('No existe equilibrio con θ ≠ 0 (|cosθ_eq| ≥ 1): el único equilibrio es θ = 0')

# Pequeñas oscilaciones: Ω² = V''(θ_eq) / (2 m l²)    (para θ_eq≠0:  Ω² = (ω² + 2k/m) sin²θ_eq)
Om2 = d2V_num(th_eq, m, l, l0, k, g, w)/(2*m*l**2)
if Om2 > 0:
    Om = np.sqrt(Om2)
    print(f'Equilibrio estable: Ω = {Om:.5f} rad/s,  T = {2*np.pi/Om:.5f} s')
else:
    Om = None
    print('Equilibrio inestable (Ω² ≤ 0)')
print(f'Estabilidad de θ=0: Ω² = {d2V_num(0.0, m, l, l0, k, g, w)/(2*m*l**2):.4f}  (>0 estable, <0 inestable)')

## GRÁFICAS

In [ ]:
#Gráficas: θ(t) frente a θ_eq
s = max(1, N // 20000)                                    # submuestreo

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax[0].plot(t[::s], theta[::s], lw=2, label='θ(t) numérica')
ax[0].axhline(th_eq, color='k', ls='--', lw=1.2, label=f'θ_eq = {th_eq:.4f} rad ({np.degrees(th_eq):.1f}°)')
if Om is not None:                                        # solución lineal alrededor de θ_eq
    th_lin = th_eq + (theta_0 - th_eq)*np.cos(Om*t) + (w_0/Om)*np.sin(Om*t)
    ax[0].plot(t[::s], th_lin[::s], 'r:', lw=1.5, label='aprox. lineal (solo válida para amplitud pequeña)')
ax[0].set(ylabel='θ (rad)', title='Ángulo del regulador y posición de equilibrio'); ax[0].legend(fontsize=8)

ax[1].plot(t[::s], theta[::s] - th_eq, color='tab:green')
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set(xlabel='t (s)', ylabel='θ − θ_eq (rad)')

for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()


#Gráficas: Lagrangiano, Energía y Hamiltoniano
L = L_num(theta, thd, *pars)
E = E_num(theta, thd, *pars)
H = H_num(theta, thd, *pars)

fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

ax[0].plot(t[::s], L[::s], color='tab:purple')
ax[0].set(ylabel='L (J)', title='Lagrangiano')

ax[1].plot(t[::s], E[::s], color='tab:blue', lw=2, label='E = T + U  (no se conserva)')
ax[1].plot(t[::s], H[::s], 'r--', lw=1.5, label='H = p_θ θ̇ − L  (se conserva)')
ax[1].set(ylabel='J', title='Energía y Hamiltoniano'); ax[1].legend(fontsize=8)

eps = 1e-18                                               # evita log(0)
ax[2].semilogy(t[::s], np.abs(H[::s] - H[0])/abs(H[0]) + eps, color='tab:red')
ax[2].set(xlabel='t (s)', ylabel='|H − H₀| / |H₀|', title='Error numérico en H')

for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
#Comprobación de Ω con amplitud pequeña:  θ0 = θ_eq + δ,  θ̇0 = 0
if Om is not None:
    delta = 0.02                                          # rad
    t_c = np.arange(0, 20, 5e-4)
    th_c, _ = Motion(th_eq + delta, 0.0, pars, t_c)
    th_lin_c = th_eq + delta*np.cos(Om*t_c)

    # Periodo numérico: cruces ascendentes de θ por θ_eq
    d = th_c - th_eq
    cr = np.where((d[:-1] < 0) & (d[1:] >= 0))[0]
    t_cr = t_c[cr] - d[cr]*(t_c[cr+1] - t_c[cr])/(d[cr+1] - d[cr])   # interpolación lineal
    T_num = np.mean(np.diff(t_cr))
    print(f'T numérico = {T_num:.5f} s   |   T teórico = {2*np.pi/Om:.5f} s   |   error = {abs(T_num - 2*np.pi/Om)/(2*np.pi/Om):.2e}')

    fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    ax[0].plot(t_c, th_c, lw=2, label='θ(t) numérica')
    ax[0].plot(t_c, th_lin_c, 'r--', lw=1.2, label='θ_eq + δ cos(Ω t)')
    ax[0].axhline(th_eq, color='k', ls=':', lw=1, label='θ_eq')
    ax[0].set(ylabel='θ (rad)', title=f'Pequeñas oscilaciones (δ = {delta} rad)'); ax[0].legend(fontsize=8)
    ax[1].plot(t_c, th_c - th_lin_c, color='tab:green')
    ax[1].set(xlabel='t (s)', ylabel='θ_num − θ_lineal')
    for a in ax: a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## ANIMACIÓN (3D + dos planos 2D)

In [ ]:
#Animación: vista 3D + dos planos 2D (lateral x-z y cenital x-y)
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 15       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 2.0      # duración de la estela de la masa (s); 0 = sin estela
n_coils = 10       # vueltas del muelle dibujado

# Submuestreo: un fotograma cada 'step' pasos de simulación
step    = max(1, round(speed / (fps * dt)))
idx     = np.arange(0, min(N, int(T_anim / dt)), step)
trail_n = int(trail_s / (dt * step))

# Posiciones en el sistema inercial (el eje gira con φ = ω t)
ths, ts = theta[idx], t[idx]
phi = w*ts
rho, zm = l*np.sin(ths), -l*np.cos(ths)
x1, y1 = rho*np.cos(phi),  rho*np.sin(phi)         # masa 1
x2, y2 = -x1, -y1                                  # masa 2 (opuesta)
zA = -2*l*np.cos(ths)                              # anillo A

def spring_xz(zA, n=n_coils, amp=0.05*l):
    """Zigzag vertical entre O (z=0) y A (z=zA)."""
    zz = np.linspace(0, zA, 2*n + 1)
    xx = amp*np.where(np.arange(2*n + 1) % 2 == 1, 1.0, -1.0)
    xx[0] = xx[-1] = 0
    return xx, zz

# Límites
R  = l*np.abs(np.sin(theta)).max()*1.15 + 1e-9
z0, z1 = zA.min()*1.1 - 0.05*l, 0.15*l

fig = plt.figure(figsize=(14, 4.8))
ax3 = fig.add_subplot(1, 3, 1, projection='3d')
axs = fig.add_subplot(1, 3, 2)                     # plano lateral x-z
axt = fig.add_subplot(1, 3, 3)                     # plano cenital x-y

ax3.set(xlim=(-R, R), ylim=(-R, R), zlim=(z0, z1), xlabel='x', ylabel='y', zlabel='z', title='Vista 3D')
ax3.set_box_aspect((2*R, 2*R, z1 - z0))
ax3.plot([0, 0], [0, 0], [z0, z1], 'k:', lw=1)     # eje de giro

axs.set(xlim=(-R, R), ylim=(z0, z1), aspect='equal', xlabel='x (m)', ylabel='z (m)', title='Plano lateral (x-z)')
axs.axvline(0, color='k', ls=':', lw=1); axs.grid(alpha=0.3)

axt.set(xlim=(-R, R), ylim=(-R, R), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Plano cenital (x-y)')
axt.grid(alpha=0.3)

# Artistas 3D
rods3,  = ax3.plot([], [], [], 'k-', lw=1.5)                       # O -> m1 -> A -> m2 -> O
spr3,   = ax3.plot([], [], [], '-', color='tab:gray', lw=1.2)
mass3,  = ax3.plot([], [], [], 'o', color='tab:red', ms=8)
ring3,  = ax3.plot([], [], [], 's', color='tab:blue', ms=7)
trail3, = ax3.plot([], [], [], '-', color='tab:red', lw=1, alpha=0.5)
ax3.plot([0], [0], [0], 'ko', ms=5)                                # punto fijo O

# Artistas plano lateral
rodsS,  = axs.plot([], [], 'k-', lw=1.5)
sprS,   = axs.plot([], [], '-', color='tab:gray', lw=1.2)
massS,  = axs.plot([], [], 'o', color='tab:red', ms=8)
ringS,  = axs.plot([], [], 's', color='tab:blue', ms=7)
trailS, = axs.plot([], [], '-', color='tab:red', lw=1, alpha=0.5)
axs.plot(0, 0, 'ko', ms=5)

# Artistas plano cenital
rodsT,  = axt.plot([], [], 'k-', lw=1.5)                           # proyección: m1 - O - m2
massT,  = axt.plot([], [], 'o', color='tab:red', ms=8)
trailT, = axt.plot([], [], '-', color='tab:red', lw=1, alpha=0.5)
axt.plot(0, 0, 'ko', ms=5)

time_txt = fig.text(0.02, 0.95, '')

def update(k):
    # polígono O -> m1 -> A -> m2 -> O
    px = [0, x1[k], 0, x2[k], 0]
    py = [0, y1[k], 0, y2[k], 0]
    pz = [0, zm[k], zA[k], zm[k], 0]
    sx, sz = spring_xz(zA[k])
    j = max(0, k - trail_n)

    rods3.set_data_3d(px, py, pz)
    spr3.set_data_3d(sx, np.zeros_like(sx), sz)
    mass3.set_data_3d([x1[k], x2[k]], [y1[k], y2[k]], [zm[k], zm[k]])
    ring3.set_data_3d([0], [0], [zA[k]])
    trail3.set_data_3d(x1[j:k+1], y1[j:k+1], zm[j:k+1])

    rodsS.set_data(px, pz)
    sprS.set_data(sx, sz)
    massS.set_data([x1[k], x2[k]], [zm[k], zm[k]])
    ringS.set_data([0], [zA[k]])
    trailS.set_data(x1[j:k+1], zm[j:k+1])

    rodsT.set_data([x1[k], 0, x2[k]], [y1[k], 0, y2[k]])
    massT.set_data([x1[k], x2[k]], [y1[k], y2[k]])
    trailT.set_data(x1[j:k+1], y1[j:k+1])

    time_txt.set_text(f't = {ts[k]:.2f} s    θ = {np.degrees(ths[k]):.1f}°')
    return rods3, spr3, mass3, ring3, trail3, rodsS, sprS, massS, ringS, trailS, rodsT, massT, trailT, time_txt

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=False)   # blit=False: necesario con ejes 3D
plt.close(fig)
HTML(anim.to_jshtml())